# Herb / Plant Image Classification — Research Pipeline

Modular, reproducible deep learning pipeline for multi-model transfer learning with evaluation, comparison, and ensemble.

## 1. Configuration

In [ ]:
# =============================================================================
# CONFIGURATION — All hyperparameters and paths in one place
# =============================================================================
import os

# Raw dataset: single folder with one subfolder per class (48-plant structure on Kaggle)
RAW_DATASET_DIR = os.environ.get(
    "RAW_HERB_DATA",
    "/kaggle/input/datasets/developerzulkarnain/48-plant-leaves-datasets/Leaves of 48 Plants Dataset",
)
# Where to write train/val/test splits (must be writable, e.g. /kaggle/working on Kaggle)
BASE_DIR = os.environ.get("HERB_DATA", "/kaggle/working/final_split_dataset")

config = {
    "IMG_SIZE": 224,
    "BATCH_SIZE": 32,
    "EPOCHS": 30,
    "SEED": 42,
    "RAW_DATASET_DIR": RAW_DATASET_DIR,
    "TRAIN_DIR": os.path.join(BASE_DIR, "train"),
    "VAL_DIR": os.path.join(BASE_DIR, "val"),
    "TEST_DIR": os.path.join(BASE_DIR, "test"),
    "MODEL_SAVE_DIR": "models",
    "MODELS": ["EfficientNetB0", "EfficientNetV2S", "MobileNetV2", "ResNet152V2"],
    "CLASS_MODE": "categorical",
}

os.makedirs(config["MODEL_SAVE_DIR"], exist_ok=True)
print("Config:", {k: v for k, v in config.items()})

## 2. Reproducibility & Imports

In [ ]:
# =============================================================================
# REPRODUCIBILITY — Set seeds for numpy, TensorFlow, and Python random
# =============================================================================
import random
import numpy as np
import tensorflow as tf

# =============================================================================
# CUDA / GPU SETUP
# =============================================================================
def configure_cuda():
    print("TensorFlow:", tf.__version__)
    print("Built with CUDA:", tf.test.is_built_with_cuda())

    gpus = tf.config.list_physical_devices("GPU")
    if not gpus:
        print("No GPU detected by TensorFlow. Training will run on CPU.")
        print("If on Kaggle: Settings -> Accelerator -> GPU, then restart runtime.")
        return False

    print(f"Detected {len(gpus)} GPU(s):", gpus)
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            # Memory growth must be set before GPUs are initialized
            pass

    logical = tf.config.list_logical_devices("GPU")
    print(f"Logical GPU(s): {len(logical)}")
    return True

USE_GPU = configure_cuda()

def set_seeds(seed=None):
    seed = seed or config["SEED"]
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["TF_DETERMINISTIC_OPS"] = "1"
    print(f"Seeds set to {seed}")

set_seeds()

# =============================================================================
# IMPORTS
# =============================================================================
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)
from tqdm.auto import tqdm

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (
    GlobalAveragePooling2D, BatchNormalization, Dense, Dropout,
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import (
    EfficientNetB0, EfficientNetV2S, MobileNetV2, ResNet152V2,
)
from tensorflow.keras.applications.efficientnet import preprocess_input as preprocess_efficientnet
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input as preprocess_efficientnetv2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as preprocess_mobilenetv2
from tensorflow.keras.applications.resnet import preprocess_input as preprocess_resnet

ImageFile.LOAD_TRUNCATED_IMAGES = True

# Optional: enable XLA JIT for performance
tf.config.optimizer.set_jit(True)
print("XLA JIT enabled:", tf.config.optimizer.get_jit())

In [ ]:
# GPU sanity check: run a small matmul and show where it executes
print("Available physical GPUs:", tf.config.list_physical_devices("GPU"))
if USE_GPU:
    with tf.device('/GPU:0'):
        a = tf.random.normal((1024, 1024))
        b = tf.random.normal((1024, 1024))
        c = tf.matmul(a, b)
    print("GPU matmul completed. Tensor shape:", c.shape)
else:
    print("Running on CPU. Enable GPU runtime/driver to use CUDA.")

## 3. Dataset Preparation

If `train/` does not exist, the raw dataset (single folder with class subfolders) is split into train/val/test (70/15/15) under `BASE_DIR`. Otherwise the existing split is used.

In [ ]:
import shutil

# Image extensions to include when splitting
_IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".gif", ".webp"}

def split_dataset_from_raw(raw_dir, output_dir, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42):
    """
    Split a single folder (raw_dir) with one subfolder per class into train/val/test
    under output_dir. Creates output_dir/train, output_dir/val, output_dir/test.
    """
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, "Ratios must sum to 1"
    rng = np.random.default_rng(seed)
    classes = sorted([d for d in os.listdir(raw_dir) if os.path.isdir(os.path.join(raw_dir, d))])
    for split_name in ("train", "val", "test"):
        os.makedirs(os.path.join(output_dir, split_name), exist_ok=True)
        for c in classes:
            os.makedirs(os.path.join(output_dir, split_name, c), exist_ok=True)

    for class_name in classes:
        class_path = os.path.join(raw_dir, class_name)
        files = [
            f for f in os.listdir(class_path)
            if os.path.isfile(os.path.join(class_path, f)) and os.path.splitext(f)[1].lower() in _IMG_EXT
        ]
        rng.shuffle(files)
        n = len(files)
        t_end = int(n * train_ratio)
        v_end = t_end + int(n * val_ratio)
        splits = [files[:t_end], files[t_end:v_end], files[v_end:]]
        for split_name, names in zip(("train", "val", "test"), splits):
            dst_dir = os.path.join(output_dir, split_name, class_name)
            for f in names:
                shutil.copy2(os.path.join(class_path, f), os.path.join(dst_dir, f))
    print(f"Split complete: {output_dir} (train/val/test) from {len(classes)} classes")

# Create train/val/test from raw dataset if they don't exist
if not os.path.exists(config["TRAIN_DIR"]):
    if not os.path.isdir(config["RAW_DATASET_DIR"]):
        raise FileNotFoundError(
            f"Train dir not found and raw dataset not found: {config['RAW_DATASET_DIR']}\n"
            "Set RAW_HERB_DATA to the path of the folder containing class subfolders (e.g. Leaves of 48 Plants Dataset)."
        )
    print("Creating train/val/test from raw dataset...")
    split_dataset_from_raw(
        config["RAW_DATASET_DIR"],
        BASE_DIR,
        train_ratio=0.7,
        val_ratio=0.15,
        test_ratio=0.15,
        seed=config["SEED"],
    )

# Quick sanity check: list classes and counts for train/val/test
def dataset_summary(train_dir, val_dir, test_dir):
    if not os.path.exists(train_dir):
        raise ValueError(f"Train directory not found: {train_dir}")

    classes = sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])

    for name, path in [("Train", train_dir), ("Val", val_dir), ("Test", test_dir)]:
        if not os.path.exists(path):
            print(f"{name}: path not found — {path}")
            continue
        counts = []
        for c in classes:
            class_path = os.path.join(path, c)
            if os.path.exists(class_path):
                num_images = len([
                    f for f in os.listdir(class_path)
                    if os.path.isfile(os.path.join(class_path, f))
                ])
                counts.append(num_images)
        print(f"{name}: {len(classes)} classes, {sum(counts)} images")

    return classes


_classes = dataset_summary(config["TRAIN_DIR"], config["VAL_DIR"], config["TEST_DIR"])

## 4. Data Cleaning

Detect and remove corrupted images using PIL; log removed files and print total cleaned.

In [ ]:
def clean_dataset(directory):
    """
    Detect corrupted images using PIL, remove unreadable files, log them, and print total cleaned.
    """
    removed = []
    total_checked = 0
    img_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".gif", ".webp"}

    for root, _, files in os.walk(directory, topdown=False):
        for f in files:
            ext = os.path.splitext(f)[1].lower()
            if ext not in img_extensions:
                continue
            path = os.path.join(root, f)
            total_checked += 1
            try:
                with Image.open(path) as img:
                    img.verify()
            except (IOError, OSError, SyntaxError) as e:
                removed.append((path, str(e)))
                try:
                    os.remove(path)
                except OSError:
                    removed.append((path, "delete failed"))

    for path, err in removed:
        print(f"Removed (corrupted): {path} — {err}")
    print(f"Clean dataset: checked {total_checked} files, removed {len(removed)}.")
    return len(removed)

# Run cleaning on train, val, test
for split_name, path in [("Train", config["TRAIN_DIR"]), ("Val", config["VAL_DIR"]), ("Test", config["TEST_DIR"])]:
    if os.path.isdir(path):
        print(f"--- {split_name} ---")
        clean_dataset(path)

## 5. Data Augmentation

Training augmentations only. Validation and test use no augmentation.

In [ ]:
def get_train_augmentation(preprocessing_function):
    """Strong augmentation pipeline for training; no augmentation for val/test."""
    return ImageDataGenerator(
        preprocessing_function=preprocessing_function,
        rotation_range=25,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.2,
        shear_range=0.1,
        horizontal_flip=True,
        brightness_range=[0.8, 1.2],
        fill_mode="nearest",
    )

def get_val_test_augmentation(preprocessing_function):
    """No augmentation for validation and test."""
    return ImageDataGenerator(preprocessing_function=preprocessing_function)

## 6. Data Generators

Reusable `create_generators(preprocessing_function)` returning train, val, and test generators via `flow_from_directory`.

In [ ]:
def create_generators(preprocessing_function):
    """
    Create train, val, and test generators with flow_from_directory.
    Returns (train_generator, val_generator, test_generator).
    """
    sz = config["IMG_SIZE"]
    batch = config["BATCH_SIZE"]
    class_mode = config["CLASS_MODE"]

    train_datagen = get_train_augmentation(preprocessing_function)
    train_generator = train_datagen.flow_from_directory(
        config["TRAIN_DIR"],
        target_size=(sz, sz),
        batch_size=batch,
        class_mode=class_mode,
        shuffle=True,
        seed=config["SEED"],
    )

    val_datagen = get_val_test_augmentation(preprocessing_function)
    val_generator = val_datagen.flow_from_directory(
        config["VAL_DIR"],
        target_size=(sz, sz),
        batch_size=batch,
        class_mode=class_mode,
        shuffle=False,
    )

    test_datagen = get_val_test_augmentation(preprocessing_function)
    test_generator = test_datagen.flow_from_directory(
        config["TEST_DIR"],
        target_size=(sz, sz),
        batch_size=batch,
        class_mode=class_mode,
        shuffle=False,
    )
    return train_generator, val_generator, test_generator

## 7. Model Building

Build transfer-learning model: ImageNet base, frozen at first; custom head (GlobalAveragePooling2D → BatchNorm → Dense(256) → Dropout(0.5) → softmax). Compiled with Adam(1e-4), categorical_crossentropy, accuracy.

In [ ]:
# Map config model names to Keras application class and preprocessing function
MODEL_MAP = {
    "EfficientNetB0": (EfficientNetB0, preprocess_efficientnet),
    "EfficientNetV2S": (EfficientNetV2S, preprocess_efficientnetv2),
    "MobileNetV2": (MobileNetV2, preprocess_mobilenetv2),
    "ResNet152V2": (ResNet152V2, preprocess_resnet),
}

def build_model(base_model_name, num_classes):
    """
    Load pretrained base (ImageNet), freeze base initially, add custom head:
    GlobalAveragePooling2D → BatchNormalization → Dense(256, relu) → Dropout(0.5) → Dense(num_classes, softmax).
    Compile with Adam(1e-4), categorical_crossentropy, accuracy.
    """
    if base_model_name not in MODEL_MAP:
        raise ValueError(f"Unknown model: {base_model_name}. Choose from {list(MODEL_MAP.keys())}")
    BaseClass, _ = MODEL_MAP[base_model_name]
    sz = config["IMG_SIZE"]

    base = BaseClass(weights="imagenet", include_top=False, input_shape=(sz, sz, 3))
    for layer in base.layers:
        layer.trainable = False

    model = Sequential([
        base,
        GlobalAveragePooling2D(),
        BatchNormalization(),
        Dense(256, activation="relu"),
        Dropout(0.5),
        Dense(num_classes, activation="softmax"),
    ])
    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

## 8. Training Pipeline

For each model: load preprocessing, create generators, build model, train with EarlyStopping(patience=5), ReduceLROnPlateau(patience=3, factor=0.3), ModelCheckpoint(save_best_only). Save best to `models/{model_name}_best.h5`.

In [ ]:
def get_callbacks(model_name):
    save_dir = config["MODEL_SAVE_DIR"]
    path = os.path.join(save_dir, f"{model_name}_best.h5")
    return [
        EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True,
            verbose=1,
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            patience=3,
            factor=0.3,
            min_lr=1e-7,
            verbose=1,
        ),
        ModelCheckpoint(
            filepath=path,
            monitor="val_loss",
            save_best_only=True,
            verbose=1,
        ),
    ]

# Train all models; store histories and trained models for later evaluation
histories = {}
trained_models = {}
class_indices = None

for model_name in config["MODELS"]:
    BaseClass, preprocess_fn = MODEL_MAP[model_name]
    print(f"\n{'='*60}\nTraining {model_name}\n{'='*60}")
    train_gen, val_gen, test_gen = create_generators(preprocess_fn)
    if class_indices is None:
        class_indices = train_gen.class_indices
    num_classes = len(class_indices)

    model = build_model(model_name, num_classes)
    callbacks = get_callbacks(model_name)
    history = model.fit(
        train_gen,
        epochs=config["EPOCHS"],
        validation_data=val_gen,
        callbacks=callbacks,
        verbose=1,
    )
    histories[model_name] = history.history
    trained_models[model_name] = (model, preprocess_fn, test_gen)

## 9. Evaluation Metrics

Per-model: test accuracy, precision, recall, F1 (sklearn); classification_report and confusion_matrix.

In [ ]:
def evaluate_model(model, test_generator):
    """Return y_true, y_pred, and dict of accuracy, precision, recall, f1 (macro)."""
    test_generator.reset()
    pred_proba = model.predict(test_generator, verbose=1)
    y_pred = np.argmax(pred_proba, axis=1)
    y_true = test_generator.classes
    labels = list(test_generator.class_indices.keys())

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=labels, zero_division=0))
    cm = confusion_matrix(y_true, y_pred)
    return y_true, y_pred, {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}, cm, labels

# Evaluate each trained model and store results
results = {}
all_cms = {}
for model_name, (model, _, test_gen) in trained_models.items():
    print(f"\n--- {model_name} ---")
    y_true, y_pred, metrics, cm, labels = evaluate_model(model, test_gen)
    results[model_name] = metrics
    all_cms[model_name] = (cm, labels)

## 10. Visualization

Research-style plots: Training vs Validation Accuracy/Loss; Confusion Matrix heatmap; Model performance bar chart.

In [ ]:
# 1) Training vs Validation Accuracy and Loss per model
os.makedirs("outputs", exist_ok=True)
for model_name, hist in histories.items():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    epochs = range(1, len(hist["accuracy"]) + 1)
    axes[0].plot(epochs, hist["accuracy"], label="Train Accuracy", marker="o", markersize=3)
    axes[0].plot(epochs, hist["val_accuracy"], label="Val Accuracy", marker="s", markersize=3)
    axes[0].set_title(f"{model_name} — Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[1].plot(epochs, hist["loss"], label="Train Loss", marker="o", markersize=3)
    axes[1].plot(epochs, hist["val_loss"], label="Val Loss", marker="s", markersize=3)
    axes[1].set_title(f"{model_name} — Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    out_path = os.path.join("outputs", f"{model_name}_train_curves.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved: {out_path}")
    plt.show()

In [ ]:
# 2) Confusion matrix heatmap (example: first model; for many classes use small font or sample)
os.makedirs("outputs", exist_ok=True)

def plot_cm_heatmap(cm, labels, title, max_classes=30, save_path=None):
    if len(labels) > max_classes:
        cm = cm[:max_classes, :max_classes]
        labels = labels[:max_classes]
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, xticklabels=labels, yticklabels=labels, annot=False, cmap="Blues", fmt="d")
    plt.title(title)
    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.xticks(rotation=90, fontsize=6)
    plt.yticks(fontsize=6)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()

name0 = config["MODELS"][0]
cm0, labels0 = all_cms[name0]
plot_cm_heatmap(
    cm0, labels0, f"Confusion Matrix — {name0}",
    save_path=os.path.join("outputs", f"{name0}_confusion_matrix.png"),
)

In [ ]:
# 3) Model performance bar chart
os.makedirs("outputs", exist_ok=True)
df_viz = pd.DataFrame(results).T
df_viz.plot(kind="bar", figsize=(10, 5), rot=0)
plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("outputs/model_comparison.png", dpi=150, bbox_inches="tight")
print("Saved: outputs/model_comparison.png")
plt.show()

## 11. Model Comparison Table

DataFrame: Model | Accuracy | Precision | Recall | F1, sorted by Accuracy.

In [ ]:
comparison_df = pd.DataFrame(results).T
comparison_df = comparison_df.sort_values("accuracy", ascending=False)
comparison_df.index.name = "Model"
print("Final results (sorted by Accuracy):")
display(comparison_df)

## 12. Ensemble Prediction

Soft voting (average of predicted probabilities) over trained models; compare ensemble accuracy to individual models.

In [ ]:
def ensemble_predict(models, generators):
    """
    models: list of (model_name, model) or dict name -> model
    generators: for each model, the corresponding test generator (same order if list).
    Returns: (y_true, y_pred_ensemble, avg_proba)
    """
    if isinstance(models, dict):
        names = list(models.keys())
        models_list = [models[n] for n in names]
    else:
        names = [m[0] for m in models]
        models_list = [m[1] for m in models]
    # Assume all use same class_indices; use first generator for labels
    first_gen = generators[0] if isinstance(generators, list) else generators[names[0]]
    first_gen.reset()
    y_true = first_gen.classes
    n_samples = len(y_true)
    n_classes = len(first_gen.class_indices)
    avg_proba = np.zeros((n_samples, n_classes))

    for i, (name, model) in enumerate(zip(names, models_list)):
        gen = generators[i] if isinstance(generators, list) else generators[name]
        gen.reset()
        p = model.predict(gen, verbose=0)
        avg_proba += p
    avg_proba /= len(models_list)
    y_pred = np.argmax(avg_proba, axis=1)
    return y_true, y_pred, avg_proba

# Build list of (name, model) and corresponding test generators
model_list = [(n, m[0]) for n, m in trained_models.items()]
gen_list = [m[2] for m in trained_models.values()]
y_true_ens, y_pred_ens, _ = ensemble_predict(model_list, gen_list)
ensemble_acc = accuracy_score(y_true_ens, y_pred_ens)
ensemble_prec = precision_score(y_true_ens, y_pred_ens, average="macro", zero_division=0)
ensemble_rec = recall_score(y_true_ens, y_pred_ens, average="macro", zero_division=0)
ensemble_f1 = f1_score(y_true_ens, y_pred_ens, average="macro", zero_division=0)
print(f"Ensemble (soft voting) test accuracy: {ensemble_acc:.4f}")
print("Individual accuracies:", {n: results[n]["accuracy"] for n in results})

# Append ensemble to comparison table
comparison_df.loc["Ensemble"] = {"accuracy": ensemble_acc, "precision": ensemble_prec, "recall": ensemble_rec, "f1": ensemble_f1}
comparison_df = comparison_df.sort_values("accuracy", ascending=False)
print("\nComparison including Ensemble:")
display(comparison_df)

## 13. Performance Optimization (tf.data)

Optional: wrap generators in tf.data.Dataset with prefetch, cache, and AUTOTUNE for faster training.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def generator_to_dataset(generator, cache=False):
    """Wrap ImageDataGenerator in tf.data.Dataset with prefetch, cache, and AUTOTUNE."""
    sz = config["IMG_SIZE"]
    num_classes = getattr(generator, "num_classes", len(generator.class_indices))
    ds = tf.data.Dataset.from_generator(
        lambda g=generator: g,
        output_signature=(
            tf.TensorSpec(shape=(None, sz, sz, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(None, num_classes), dtype=tf.float32),
        ),
    )
    ds = ds.prefetch(AUTOTUNE)
    if cache:
        ds = ds.cache()
    return ds

# Example: use in training (optional — replace train_gen in model.fit with ds)
# train_ds = generator_to_dataset(train_gen, cache=True)
# model.fit(train_ds, epochs=config["EPOCHS"], validation_data=val_gen, callbacks=callbacks)

## 14. Optional Research Features

Grad-CAM visualization, Test-Time Augmentation (TTA), and learning-rate scheduling.

In [ ]:
# --- Grad-CAM (model explainability) ---
# Last conv block output before global pooling (Keras Applications naming)
LAST_CONV_LAYERS = {
    "EfficientNetB0": "top_activation",
    "EfficientNetV2S": "top_activation",
    "MobileNetV2": "out_relu",
    "ResNet152V2": "post_relu",
}

def make_gradcam_model(model, last_conv_layer_name):
    """
    Grad-CAM model: outputs (class logits, last conv feature map).
    Keras 3: `model(inp)` + `last_conv.output` can be two disconnected graphs → gradients None.
    Warm up the Sequential, then use model.input with [model.output, last_conv.output] on one graph.
    """
    base = model.layers[0]
    last_conv = base.get_layer(last_conv_layer_name)
    in_shape = model.layers[0].input_shape
    h, w = in_shape[1], in_shape[2]
    c = in_shape[3] if len(in_shape) > 3 else 3
    if h is None or w is None:
        rz = int(config["IMG_SIZE"])
        sz = (rz, rz, int(c) if c is not None else 3)
    else:
        sz = (int(h), int(w), int(c) if c is not None else 3)
    dummy = np.zeros((1,) + sz, dtype=np.float32)
    model(dummy, training=False)

    inp = model.input
    conv_tensor = last_conv.output
    return keras.Model(inputs=inp, outputs=[model.output, conv_tensor])

def gradcam_heatmap(img_array, grad_model, pred_index=0):
    img_tensor = tf.cast(np.expand_dims(img_array, 0), tf.float32)
    with tf.GradientTape() as tape:
        preds, conv_output = grad_model(img_tensor, training=False)
        conv_output = tf.cast(conv_output, tf.float32)
        tape.watch(conv_output)
        preds_f = tf.cast(preds, tf.float32)
        # Scalar class score so the tape has a well-defined gradient path
        loss = preds_f[0, pred_index]
    grads = tape.gradient(loss, conv_output)
    if grads is None:
        raise ValueError(
            "Gradients are None — try another LAST_CONV_LAYERS name, "
            "or ensure the grad model was built after model(dummy, training=False)."
        )
    pooled_grads = tf.reduce_mean(grads, axis=(1, 2))
    heatmap = tf.reduce_sum(conv_output[0] * pooled_grads[0], axis=-1)
    heatmap = heatmap.numpy()
    heatmap = np.maximum(heatmap, 0)
    heatmap = heatmap / (np.max(heatmap) + 1e-8)
    return heatmap

def overlay_gradcam(original_img, heatmap, alpha=0.4, save_path=None):
    """Overlay Grad-CAM heatmap on original image (RGB uint8 or float 0–1)."""
    try:
        import cv2
    except ImportError:
        raise ImportError("Install opencv-python for overlay_gradcam: pip install opencv-python")

    heatmap_uint8 = np.uint8(255 * heatmap)
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    h, w = original_img.shape[:2]
    heatmap_resized = cv2.resize(heatmap_color, (w, h))
    orig = original_img.copy()
    if orig.max() <= 1.0:
        orig = (orig * 255).astype(np.uint8)
    else:
        orig = orig.astype(np.uint8)
    superimposed = cv2.addWeighted(orig, 1 - alpha, heatmap_resized, alpha, 0)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(orig)
    axes[0].set_title("Original")
    axes[0].axis("off")
    axes[1].imshow(heatmap_color)
    axes[1].set_title("Grad-CAM heatmap")
    axes[1].axis("off")
    axes[2].imshow(superimposed)
    axes[2].set_title("Overlay")
    axes[2].axis("off")
    plt.tight_layout()
    os.makedirs("outputs", exist_ok=True)
    if save_path is None:
        save_path = os.path.join("outputs", "gradcam_overlay.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    print(f"Saved: {save_path}")
    plt.show()

# --- Test-Time Augmentation (TTA) ---
def predict_tta(model, image, datagen, n_tta=5):
    """Average predictions over n_tta augmented copies of image."""
    aug = datagen.flow(np.expand_dims(image, 0), batch_size=1)
    preds = []
    for _ in range(n_tta):
        x = next(aug)[0]
        preds.append(model.predict(np.expand_dims(x, 0), verbose=0))
    return np.mean(preds, axis=0)

# --- Learning rate schedule (e.g. cosine decay) ---
def get_cosine_schedule(epochs, initial_lr=1e-4):
    return keras.optimizers.schedules.CosineDecay(initial_lr, epochs)

# Run Grad-CAM demo when training has finished (set False to only define helpers)
RUN_GRADCAM = True

if RUN_GRADCAM and "trained_models" in dir() and isinstance(trained_models, dict) and len(trained_models) > 0:
    model_name = config["MODELS"][0]
    if model_name not in LAST_CONV_LAYERS:
        print(f"Grad-CAM: no layer mapping for {model_name}, skipping.")
    else:
        model, _, test_gen = trained_models[model_name]
        test_gen.reset()
        x_batch, _ = next(test_gen)
        img = x_batch[0]
        grad_m = make_gradcam_model(model, LAST_CONV_LAYERS[model_name])
        pred_idx = int(np.argmax(model.predict(np.expand_dims(img, 0), verbose=0)[0]))
        hm = gradcam_heatmap(img, grad_m, pred_index=pred_idx)
        overlay_gradcam(img, hm, save_path="outputs/gradcam_overlay.png")
elif RUN_GRADCAM:
    print("Grad-CAM skipped: run the training pipeline cell first (so `trained_models` exists).")